In [ ]:
from datetime import timedelta
import datetime
import io
from PIL import Image
from cairosvg import svg2png
import numpy as np
from glider import GoProVideoMetadata, IGCFile
import cv2
from zoneinfo import ZoneInfo

In [ ]:
video_filename = 'GX010108.MP4'
# out_filename = 'GX020107_enriched.mp4'
out_filename_no_audio = 'noaudio.mp4'
igc_data = IGCFile("2026-05-01-XTR-A68A17562166-01.IGC")
# TODO: check start time of video when cut by gopro (2xx)
meta0 = GoProVideoMetadata.from_video("GX010107.MP4")
gopro_meta = GoProVideoMetadata.from_video(video_filename)
gopro_meta.creation_time -= timedelta(hours=2)

In [ ]:

gopro_meta = GoProVideoMetadata.from_video(video_filename)

In [ ]:
# Compass Data
# compass_bg = 
# svg_data = open("compass-north-svgrepo-com.svg", "rb").read()
# png_data = svg2png(bytestring=svg_data, output_width=100)

compass_position = (10,10)
compass_background = Image.open("compass_background.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle = Image.open("compass_needle.png").convert("RGBA").resize((200, 200), Image.Resampling.LANCZOS)
needle_center = [x/2 for x in needle.size]

In [ ]:
def draw_compass(frame: np.array, heading: float):
    img = Image.fromarray(frame)
    rotated_overlay = needle.rotate(-heading, center=needle_center, resample=Image.BICUBIC)
    img.paste(compass_background, compass_position, compass_background)
    img.paste(rotated_overlay, compass_position, rotated_overlay)
    frame = np.array(img)
    return frame

def draw_altitude_line(igc: IGCFile, time: datetime.datetime, frame):
    backwards, forwards = igc.get_altitude_progression(time)
    y = [*-backwards, *-forwards] # Invert the values (image pixel direction)
    y += min(y) # Make values positive
    x = [x for x in range(len(y))]
    x_image_size = 280
    y_image_size = 100
    plot_position = (240, 90)
    x_scaled = plot_position[0] + (x - np.min(x)) / (np.max(x) - np.min(x)) * x_image_size
    y_scaled = plot_position[1] + (y - np.min(y)) / (np.max(y) - np.min(y)) * y_image_size

    x = x_scaled
    y = y_scaled

    points = np.column_stack((x, y)).astype(np.int32)

    # 3. Draw the Fading Shadow (Negative Y direction)
    # In image coordinates, "down" is the positive Y-axis.
    shadow_layers = 25
    max_alpha = 0.2

    for i in range(shadow_layers, 0, -1):
        # Create a transparent overlay for the current shadow layer
        overlay = frame.copy()
        
        # Calculate transparency: fades out as it moves away from the line
        alpha = max_alpha * (1 - i / shadow_layers)**1.5
        
        # Offset the points downward to create the shadow effect
        offset_points = points.copy()
        offset_points[:, 1] += i
        
        # Draw the shadow line on the overlay
        cv2.polylines(overlay, [offset_points], isClosed=False, color=(75, 150, 200), thickness=5)
        
        # Blend the overlay back into the main image
        frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # 4. Draw the main curve on top
    cv2.polylines(frame, [points[0:int(len(backwards))]], isClosed=False, color=(0, 0, 0), thickness=4, lineType=cv2.LINE_AA)
    cv2.polylines(frame, [points[int(len(backwards)):]], isClosed=False, color=(0, 255, 0), thickness=4, lineType=cv2.LINE_AA)
    return frame

In [ ]:
start_datetime = gopro_meta.creation_time - timedelta(hours=2)
cap = cv2.VideoCapture(video_filename)

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_filename_no_audio, fourcc, fps, (width, height))
frame_count = meta0.number_of_frames

In [ ]:
current_frame_time = start_datetime + timedelta(seconds=frame_count/fps)
heading_value = igc_data.get_bearing_at_time(current_frame_time).bearing_deg
heading_value, current_frame_time

In [ ]:

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    current_frame_time = start_datetime + timedelta(seconds=frame_count/fps)
    heading_value = igc_data.get_bearing_at_time(current_frame_time).bearing_deg
    altitude_text = f"ALT {igc_data.get_gps_altitude_at_time(current_frame_time)} m"
    print(heading_value)

    frame = cv2.putText(
        frame,
        current_frame_time.replace(microsecond=0).astimezone(ZoneInfo("Europe/Zurich")).isoformat(),
        (25, frame.shape[0]-25), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
    frame = cv2.putText(
        frame,
        altitude_text,
        (240, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    frame = draw_compass(frame, heading_value)
    frame = draw_altitude_line(igc_data, current_frame_time, frame)
    out.write(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_count += 1

cap.release()
out.release()
print("Export Complete!")

In [ ]:
# ffmpeg -i noaudio.mp4 -i GX020107.MP4 -c:v "copy" -c:a "aac" -map "0:v:0" -map "1:a:0" -shortest test.mp4